# 141 — Prompt Caching

**What this workbook teaches:**
- How Anthropic's `cache_control` parameter works and when caching activates (≥1024 tokens)
- The difference between regular and cache-enabled API call shapes
- How to measure token savings and latency improvements from prompt caching
- How to use the LangGraph workflow to benchmark uncached vs. cached calls

---

**Two sections:**
- **Part A** — Pure Python, no API key required. Simulate and inspect the caching mechanics.
- **Part B** — Live Anthropic calls. Requires `ANTHROPIC_API_KEY` in your environment.

In [ ]:
%pip install -q anthropic python-dotenv langgraph

---
## Part A — Concepts (no API key needed)

### What is prompt caching?

Anthropic's prompt caching lets you mark a part of your request — typically a large system prompt — so the model can reuse a pre-computed prefix on follow-up calls.

**How it works:**
1. You add `"cache_control": {"type": "ephemeral"}` to the system prompt block.
2. On the **first** call, Anthropic processes and stores the prefix. You pay full input token price.
3. On **subsequent** calls with the same prefix, Anthropic reads from the cache. You pay the (lower) cache-read price instead of the full input price.

**When does caching activate?**
- The cacheable prefix must be **≥ 1024 tokens** (roughly 4000 characters).
- Cache entries are **ephemeral** by default — they persist for up to 5 minutes of inactivity.
- Only the system prompt (or a large tool definition) is typically cache-worthy; the user message changes every call.

**Pricing impact (Haiku, as of 2025):**

| Token type | Price per MTok |
|---|---|
| Regular input | \$0.25 |
| Cache write | \$0.30 (slightly more) |
| Cache read | \$0.03 (10× cheaper) |

The savings compound when the same large prompt is reused across many queries.

In [ ]:
# Part A — Cell 2: Simulate token savings with mock data
# This mirrors the compute_savings() function in src/tools.py exactly.

def compute_savings(uncached: list[dict], cached: list[dict]) -> dict:
    """Compute token and cost savings between uncached and cached runs."""
    cost_per_mtok = 0.25  # Haiku input price $/MTok
    uncached_tok = sum(r["input_tokens"] for r in uncached)
    # Billed tokens for cached calls = input_tokens - cache_read_tokens
    cached_tok = sum(r["input_tokens"] - r["cache_read"] for r in cached)
    saved_tok = uncached_tok - cached_tok
    return {
        "uncached_total_input_tokens": uncached_tok,
        "cached_total_billed_tokens": cached_tok,
        "tokens_saved": saved_tok,
        "estimated_savings_usd": round(saved_tok / 1_000_000 * cost_per_mtok, 6),
        "avg_latency_uncached_ms": round(sum(r["latency_ms"] for r in uncached) / len(uncached)),
        "avg_latency_cached_ms": round(sum(r["latency_ms"] for r in cached) / len(cached)),
    }


# Mock data: 3 questions, system prompt ~1500 tokens each
MOCK_UNCACHED = [
    {"input_tokens": 1520, "cache_read": 0, "latency_ms": 820},
    {"input_tokens": 1518, "cache_read": 0, "latency_ms": 790},
    {"input_tokens": 1522, "cache_read": 0, "latency_ms": 810},
]

# Cached calls: same total tokens, but 1400 of them are served from cache
MOCK_CACHED = [
    {"input_tokens": 1520, "cache_read": 1400, "latency_ms": 210},
    {"input_tokens": 1518, "cache_read": 1400, "latency_ms": 195},
    {"input_tokens": 1522, "cache_read": 1400, "latency_ms": 205},
]

savings = compute_savings(MOCK_UNCACHED, MOCK_CACHED)

print("=== Simulated Savings (mock data) ===")
for k, v in savings.items():
    print(f"  {k}: {v}")

print()
pct = round(savings["tokens_saved"] / savings["uncached_total_input_tokens"] * 100, 1)
lat_improvement = round(
    (savings["avg_latency_uncached_ms"] - savings["avg_latency_cached_ms"])
    / savings["avg_latency_uncached_ms"] * 100, 1
)
print(f"Token reduction:   {pct}%")
print(f"Latency reduction: {lat_improvement}%")

### The two API call shapes

The only difference between a regular and a cache-enabled call is how the `system` parameter is formatted.

**Regular call** — system is a plain string:
```python
client.messages.create(
    model="claude-haiku-4-5-20251001",
    system="You are an expert...",   # <-- plain string
    messages=[{"role": "user", "content": question}],
)
```

**Cache-enabled call** — system is a list with `cache_control`:
```python
client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=[
        {
            "type": "text",
            "text": "You are an expert...",
            "cache_control": {"type": "ephemeral"}  # <-- this flag is all that changes
        }
    ],
    messages=[{"role": "user", "content": question}],
)
```

The response then includes `usage.cache_read_input_tokens` telling you how many tokens came from cache.

In [ ]:
# Part A — Cell 4: Build the two message shapes as Python dicts (no API call)
import json

# Construct a large system prompt (>1024 tokens ~ >4000 chars)
SYSTEM_PROMPT = (
    "You are an expert on the Python programming language with deep knowledge of: "
    "language specification and PEPs, standard library internals, "
    "CPython implementation details, performance optimization techniques, "
    "concurrency patterns (threading, asyncio, multiprocessing), "
    "packaging, testing, and deployment best practices. "
    "Answer questions accurately and concisely. "
    + ("This system prompt is intentionally long to exceed the 1024-token caching threshold. " * 25)
)

rough_token_count = len(SYSTEM_PROMPT) // 4
print(f"System prompt length: {len(SYSTEM_PROMPT)} chars (~{rough_token_count} tokens)")
print(f"Exceeds 1024-token threshold: {rough_token_count >= 1024}")
print()

question = "What is the GIL and how does it affect multithreading?"

# Shape 1: Regular (no caching)
regular_call = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 256,
    "system": SYSTEM_PROMPT,          # plain string
    "messages": [{"role": "user", "content": question}],
}

# Shape 2: Cache-enabled
cached_call = {
    "model": "claude-haiku-4-5-20251001",
    "max_tokens": 256,
    "system": [                        # list with cache_control block
        {
            "type": "text",
            "text": SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"},
        }
    ],
    "messages": [{"role": "user", "content": question}],
}

print("--- Regular call: system field type ---")
print(f"  type(system): {type(regular_call['system']).__name__}")
print()
print("--- Cached call: system field type and structure ---")
print(f"  type(system): {type(cached_call['system']).__name__} of {len(cached_call['system'])} block(s)")
print("  Block keys:", list(cached_call["system"][0].keys()))
print("  cache_control:", json.dumps(cached_call["system"][0]["cache_control"]))

---
## Part B — Live Anthropic calls

**Requires:** `ANTHROPIC_API_KEY` in your `.env` file or environment.

This section runs the full LangGraph workflow:
1. Sends 3 questions **without** caching — measures tokens billed and latency.
2. Sends the same 3 questions **with** `cache_control` — measures cache hits and latency.
3. Prints a comparison table.

> Note: the first cached call writes the cache (slightly higher latency); subsequent calls benefit from it.

In [ ]:
# Part B — Setup: load key, fail fast
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is not set. "
        "Add it to your .env file or export it before running Part B."
    )
print("ANTHROPIC_API_KEY loaded.")

In [ ]:
# Part B — Run uncached vs cached benchmark via the LangGraph workflow
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "."))

import anthropic
from src.workflow import create_workflow

SYSTEM_PROMPT_LIVE = (
    "You are an expert on the Python programming language with deep knowledge of: "
    "language specification and PEPs, standard library internals, "
    "CPython implementation details, performance optimization techniques, "
    "concurrency patterns (threading, asyncio, multiprocessing), "
    "packaging, testing, and deployment best practices. "
    "Answer questions accurately and concisely. "
    + ("This system prompt is intentionally long to exceed the 1024-token caching threshold. " * 25)
)

QUESTIONS = [
    "What is the GIL and how does it affect multithreading?",
    "Explain the difference between __str__ and __repr__.",
    "How does Python's garbage collector handle circular references?",
]

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
workflow = create_workflow()
config = {"configurable": {"client": client}}

result = workflow.invoke({
    "system_prompt": SYSTEM_PROMPT_LIVE,
    "questions": QUESTIONS,
    "uncached_results": [],
    "cached_results": [],
    "savings": {},
}, config=config)

print("=== Cache Benchmark Results ===\n")
print(f"{'Q#':<4} {'Uncached tokens':>16} {'Uncached ms':>12} {'Cached billed':>14} {'Cache read':>11} {'Cached ms':>10}")
print("-" * 72)
for i, (unc, cac) in enumerate(zip(result["uncached_results"], result["cached_results"])):
    print(f"Q{i+1:<3} {unc['input_tokens']:>16} {unc['latency_ms']:>12} "
          f"{cac['input_tokens'] - cac['cache_read']:>14} {cac['cache_read']:>11} {cac['latency_ms']:>10}")

print()
s = result["savings"]
print(f"Tokens saved:         {s['tokens_saved']}")
print(f"Estimated savings:    ${s['estimated_savings_usd']}")
print(f"Avg latency uncached: {s['avg_latency_uncached_ms']} ms")
print(f"Avg latency cached:   {s['avg_latency_cached_ms']} ms")